# Cross-sensory 3D prototype alignment — Colab

This notebook runs only in a Colab GPU runtime. It uses curated ChemTastesDB labels for the main taste task and keeps FlavorDB taste wording as weak auxiliary evidence.

Before running: push/upload the current project version containing `src/dataset/sensory.py`, `src/sensory/`, and `scripts/train_cross_sensory.py`.

In [ ]:
%pip install -q unimol-tools pandas pyarrow scikit-learn rdkit openpyxl

import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

## Get the current project

Replace `REPO_URL` if needed. The upstream repository must include the new cross-sensory files; cloning an older commit will fail the guard below.

In [ ]:
from pathlib import Path
import subprocess, sys

REPO_URL = 'https://github.com/FufanLu/molecular-foundation-model.git'  # replace with your pushed branch/repository
repo_dir = Path('/content/molecular-foundation-model')
if not repo_dir.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only'], check=True)
print('project commit:', subprocess.check_output(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], text=True).strip())
sys.path.insert(0, str(repo_dir))
required_code = [repo_dir / 'src/dataset/sensory.py', repo_dir / 'src/sensory/model.py', repo_dir / 'scripts/train_cross_sensory.py']
missing_code = [str(path) for path in required_code if not path.exists()]
if missing_code:
    raise RuntimeError('This clone is missing the cross-sensory implementation. Push/upload the current project first: ' + ', '.join(missing_code))
%cd /content/molecular-foundation-model

## Upload the four raw files

This cell opens Colab's file picker only when a required file is absent. Select all four files together: `leffingwell_merged.csv`, `goodscents_merged.csv`, `flavordb_merged.csv`, and `ChemTastesDB_database.xlsx`.

In [ ]:
from google.colab import files
import shutil

raw_dir = Path('data/raw')
raw_dir.mkdir(parents=True, exist_ok=True)
required_data = {'leffingwell_merged.csv', 'goodscents_merged.csv', 'flavordb_merged.csv', 'ChemTastesDB_database.xlsx'}
missing_data = required_data - {path.name for path in raw_dir.iterdir()}
if missing_data:
    print('Upload:', sorted(missing_data))
    uploaded = files.upload()
    for filename in uploaded:
        if filename in required_data:
            shutil.move(filename, raw_dir / filename)
missing_data = required_data - {path.name for path in raw_dir.iterdir()}
if missing_data:
    raise FileNotFoundError(f'Missing required raw files: {sorted(missing_data)}')
print('Raw data ready:', sorted(required_data))

## Build the strong/weak sensory dataset

`taste_labels` are ChemTastesDB-curated core labels: sweet, bitter, and umami. Sour and salty remain in the dataset as separate low-shot endpoints. The 3D molecule encoder aligns to learned sensory-label prototypes; strong/weak pairs align odor-label and taste-label sets rather than two projections of the same molecule. FlavorDB-derived terms remain weak evidence only.

In [ ]:
!python -m src.dataset.sensory --raw-dir data/raw --chem-tastes data/raw/ChemTastesDB_database.xlsx --output-dir data/processed/sensory --n-splits 5

import json
audit = json.loads(Path('data/processed/sensory/audit.json').read_text())
print('curated taste labels:', audit['taste_label_counts'])
print('core taste-labelled molecules:', audit['core_taste_labeled_molecule_count'])
print('exact odor–taste pairs:', audit['paired_molecule_count'])
print('low-shot sour/salty examples:', audit['sour_molecule_count'], audit['salty_molecule_count'])
print('fold summary:', json.dumps(audit['fold_summary'], indent=2))

## Train one fold first

This uses fold 0 as test, fold 1 as validation, and the remaining folds for training. The sampler places at least two exact pairs in every training batch so label-set InfoNCE is non-zero. Checkpoints and JSON metrics are written under `outputs/v3_prototype_d/`.

The first run featurises every molecule once and caches the Uni-Mol inputs as `data/processed/sensory/unimol_features_<digest>.pkl`; later folds load the cache instead of re-featurising. Per-label decision thresholds are selected on the validation split and locked before the single test evaluation.

In [ ]:
!PYTHONPATH=/content/molecular-foundation-model:$PYTHONPATH python scripts/train_cross_sensory.py --data data/processed/sensory/molecules.parquet --output-dir outputs/v3_prototype_d --folds 0 --epochs 30 --patience 6 --batch-size 16 --lora-layers 4 --lora-rank 4 --paired-per-batch 2 --weak-paired-per-batch 2 --prototype-weight 0.05 --contrastive-weight 0.05 --weak-taste-weight 0.02 --weak-contrastive-weight 0.01 --weak-temperature 0.5

## Run all five folds

After fold 0 is stable, launch every fold in one process. Validation is fold `(test + 1) % 5`, the cached Uni-Mol inputs are reused, and folds whose metrics JSON already exist are skipped, so an interrupted run resumes cheaply. The main taste task is sweet, bitter, and umami; sour and salty remain separate low-shot endpoints and are not included in training or the headline taste macro-F1.

Each `fold*_metrics.json` reports validation-selected-threshold test F1, a fixed-0.5 reference, and pair retrieval probes (Recall@1/5, MRR) for the alignment claim. Aggregate all five folds afterwards with `scripts/aggregate_cross_sensory.py`.

In [ ]:
!PYTHONPATH=/content/molecular-foundation-model:$PYTHONPATH python scripts/train_cross_sensory.py --data data/processed/sensory/molecules.parquet --output-dir outputs/v3_prototype_d --folds 0,1,2,3,4 --epochs 30 --patience 6 --batch-size 16 --lora-layers 4 --lora-rank 4 --paired-per-batch 2 --weak-paired-per-batch 2 --prototype-weight 0.05 --contrastive-weight 0.05 --weak-taste-weight 0.02 --weak-contrastive-weight 0.01 --weak-temperature 0.5

## Aggregate all five folds

Run this only after `fold0_metrics.json` through `fold4_metrics.json` all exist under `outputs/v3_prototype_d/`. The aggregator writes a machine-readable `summary.json` and a Markdown table with held-out mean ± standard deviation, and rejects mixed task definitions, alignment settings, or duplicate test folds. Download `reports/v3_prototype_d_5fold/` and commit it to the repository for provenance.

In [ ]:
!python scripts/aggregate_cross_sensory.py --metrics outputs/v3_prototype_d/fold*_metrics.json --output-dir reports/v3_prototype_d_5fold